In [ ]:
from main import*
from run_estimator import*
from circle_utilities import*
from datasets import two_balls_line

Single Linkage algorithm

In [ ]:
def single_linkage(W, n_clusters):
    N = W.shape[0]
    parent = np.arange(N)
    size = np.ones(N, dtype=int)

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(x, y):
        rx, ry = find(x), find(y)
        if rx == ry:
            return False
        if size[rx] < size[ry]:
            rx, ry = ry, rx
        parent[ry] = rx
        size[rx] += size[ry]
        return True

    i, j = np.triu_indices(N, k=1)
    w = W[i, j]
    mask = w != 0
    i, j, w = i[mask], j[mask], w[mask]

    order = np.argsort(w)
    merges = []
    n_clusters_current = N

    for idx in order:
        if n_clusters_current <= n_clusters:
            break
        a, b = i[idx], j[idx]
        if union(a, b):
            merges.append((a, b, w[idx]))
            n_clusters_current -= 1

    labels = np.array([find(x) for x in range(N)])
    _, labels = np.unique(labels, return_inverse=True)
    return labels, merges

def relabel(A,i,v):  # Permutes the values in A such that A[i] = v. Assumes A only contains nonnegative integers
    assert v in A
    if A[i] == v : return A
    v0 = A[i]
    T = np.arange(A.max() + 1)
    T[v], T[v0] = v0, v
    return T[A]

cmap = plt.cm.tab10(np.arange(20) % 10)
def plot_clusters(X, labels):
    fig, ax = plt.subplots()

    ax.scatter(X[:, 0], X[:, 1], c=cmap[labels], s=20, zorder=3)
    ax.axis('off')
    ax.set_aspect('equal', adjustable='box')
    return fig

Dataset

In [ ]:
n_samples = 2000
proportion = 2 / n_samples**0.5

seed = 16  # Random seed

X_init = np.array([[-2,0],[2,0]])
X = two_balls_line(n_samples, proportion, X_init=X_init, seed=seed)

In [ ]:
color_points = 'gray'
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(X[:,0], X[:,1], c=color_points, s=20, marker='x')
ax.axis('off')
ax.set_aspect('equal', adjustable='box')

plt.show()

FDTM

In [ ]:
m = proportion
p = 2
beta = 4
dtm_arg = DTM_arg(m, p, beta)
dtm = DTM(X, dtm_arg)

precision = 5

In [ ]:
M = FDTM(X, dtm)

M.select_edges(knn = 40)  # Hardcoded: we only need small edges in practice
M.make_edges(precision_segment=precision, runtime=True)

M.weight_matrix = nx.to_numpy_array(M.G, weight='weight')

In [ ]:
k_clusters = np.array([20, 10, 5, 2]) 

Plots

In [ ]:
save = True  # Save figures

In [ ]:
for k in k_clusters:
    labels, _ = single_linkage(M.weight_matrix, k)
    labels = relabel(labels, 0, 0)
    if labels[1] != 0 : labels = relabel(labels, 1, 1)
    fig = plot_clusters(X, labels)
    
    if save : fig.savefig(f"figures\\cluster\\fdtm_{k}.png", bbox_inches='tight', transparent=True, dpi=300)

Euclidean / Fermat

In [ ]:
M2 = Euclidean(X)

M2.select_edges(knn=40)  # Hardcoded to go faster: we only need small edges in practice
M2.make_edges()

M2.weight_matrix = nx.to_numpy_array(M2.G, weight='weight')

In [ ]:
for k in k_clusters:
    labels, _ = single_linkage(M2.weight_matrix, k)
    labels = relabel(labels, 0, 0)
    if labels[1] != 0 : labels = relabel(labels, 1, 1)
    fig = plot_clusters(X, labels)

    if save : fig.savefig(f"figures\\cluster\\fermat_{k}.png", bbox_inches='tight', transparent=True, dpi=300)